# Lab 3 — Blocks, Blockchain and UTXO
Université Mohammed V de Rabat — Master IGOV — Blockchain: Foundations & Applications

This notebook explains and demonstrates a working implementation that follows the lab statement.


## Part 1 — Blocks and Blockchain

**Goal:** Store messages as blocks in a blockchain and secure it with Proof-of-Work.

**Block fields:** `index`, `previous_hash`, `data`, `nonce`, `hash`.

**Hash formula (as required):**
`Hash = SHA256(index + previous_hash + SHA256(data) + nonce)`


In [13]:
from src.blockchain_utxo import Blockchain

bc = Blockchain(difficulty=3)
bc.chain.append(bc.create_genesis_block())
bc.add_block('msg1' + 'msg2')
bc.add_block('msg3' + 'msg4')
bc.is_valid(), len(bc.chain), bc.chain[-1].hash

(True, 3, '0008e45a98849a3038b9326cf90fc606f55594656c617dacea8ea9f1efc6f969')

### Verification

A blockchain is valid if:
- each block hash is correct (recomputed hash matches stored hash)
- each block points to the previous block by `previous_hash`
- each block satisfies PoW difficulty (leading zeros)


In [14]:
[(b.index, b.previous_hash, b.nonce, b.hash[:12]) for b in bc.chain]

[(0, None, 289, '000450f4c0af'),
 (1,
  '000450f4c0af1b38c678bbb7b13db087619d3d87cc8ec3ec07ca72cb8c21b08b',
  6824,
  '0005f7e8bb60'),
 (2,
  '0005f7e8bb607a8dc8a6fe015ce9b8bf8df75762b9dd796e915f2667f1515b47',
  1563,
  '0008e45a9884')]

## Part 2 — Transactions and UTXO

**Transaction model:**
- `inputs`: list of `(preTxHash, outIndex)`
- `outputs`: list of `(index, value, publicKey)`
- one signature for the whole transaction (assumption: all inputs belong to same owner)

**UTXO set:** tracks all *unspent* outputs.


In [15]:
from src.blockchain_utxo import Transaction, TxInput, TxOutput, UTXOSet, validate_transaction, apply_transaction
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import serialization

# Create keypair for Alice
priv = ec.generate_private_key(ec.SECP256K1())
pub = priv.public_key()
priv_pem = priv.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
).decode('utf-8')
pub_pem = pub.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
).decode('utf-8')

utxos = UTXOSet()
# Insert one UTXO for Alice (like a coinbase)
coinbase_hash = 'COINBASE_DEMO'
utxos.add_output(coinbase_hash, TxOutput(index=0, value=500, public_key_pem=pub_pem))

# Alice spends 500 -> 200 to Bob, 300 change to Alice
bob_pub_pem = pub_pem  # demo
tx = Transaction(
    inputs=[TxInput(pre_tx_hash=coinbase_hash, out_index=0)],
    outputs=[
        TxOutput(index=0, value=200, public_key_pem=bob_pub_pem),
        TxOutput(index=1, value=300, public_key_pem=pub_pem),
    ],
)
tx.sign(priv_pem)

ok, reason = validate_transaction(tx, utxos, owner_public_key_pem=pub_pem)
ok, reason

(True, 'Valid')

### Applying the transaction

If valid, we:
- remove (spend) the referenced UTXOs
- add the new outputs as UTXOs under the new `tx_hash`


In [16]:
if ok:
    apply_transaction(tx, utxos)
utxos.snapshot()

{'fa7cb0d8945e09333d7d41d37a2b142f1376a65a68f8c94b40ea7849ff503377:0': {'index': 0,
  'value': 200,
  'public_key_pem': '-----BEGIN PUBLIC KEY-----\nMFYwEAYHKoZIzj0CAQYFK4EEAAoDQgAESgAFBRRMCtMIG2/9w+SZitu4X1fpO3a+\nL8nu1I153wJBz85kybrb8HhtZ5Bd2M7IgGBYIbS9uHomyoYI1NNKqA==\n-----END PUBLIC KEY-----\n'},
 'fa7cb0d8945e09333d7d41d37a2b142f1376a65a68f8c94b40ea7849ff503377:1': {'index': 1,
  'value': 300,
  'public_key_pem': '-----BEGIN PUBLIC KEY-----\nMFYwEAYHKoZIzj0CAQYFK4EEAAoDQgAESgAFBRRMCtMIG2/9w+SZitu4X1fpO3a+\nL8nu1I153wJBz85kybrb8HhtZ5Bd2M7IgGBYIbS9uHomyoYI1NNKqA==\n-----END PUBLIC KEY-----\n'}}

## Conclusion

This implementation follows the lab requirements and provides:
- a working PoW blockchain storing concatenated messages
- a UTXO transaction model with all required validation rules
- a minimal demo to test correctness
